In [40]:
from elk_generalization.datasets.sciq_dataset import SciQDataset
from elk_generalization.datasets.race_dataset import RaceDataset
from elk_generalization.datasets.popqa_dataset import PopQADataset


dataset_types = [SciQDataset, PopQADataset]
# dataset_types = [SciQDataset, RaceDataset, PopQADataset][::-1]
for dataset_type in dataset_types:
    ds = dataset_type("weak_lm_datases")

    print(f"Dataset: {dataset_type.__name__}")

    pythia_suite = [f"EleutherAI/pythia-{size}" for size in ["410m", "1b", "1.4b", "2.8b", "6.9b", "12b"]]
    ds.generate_quirky_dataset(weak_model_name="EleutherAI/pythia-410m", difficulty_model_names=pythia_suite, push_to_hub=True, n_train=-1, n_val=1000, n_test=1000, verbose=True)


Map:   0%|          | 0/13679 [00:00<?, ? examples/s]

Dataset: SciQDataset
Loading results from weak_lm_datases/quirky_sciq/pythia-410m_results
Loading results from weak_lm_datases/quirky_sciq/pythia-14m_results
Loading results from weak_lm_datases/quirky_sciq/pythia-70m_results
Loading results from weak_lm_datases/quirky_sciq/pythia-160m_results
Loading results from weak_lm_datases/quirky_sciq/pythia-410m_results
Loading results from weak_lm_datases/quirky_sciq/pythia-1b_results
Loading results from weak_lm_datases/quirky_sciq/pythia-1.4b_results
Loading results from weak_lm_datases/quirky_sciq/pythia-2.8b_results
Loading results from weak_lm_datases/quirky_sciq/pythia-6.9b_results
Loading results from weak_lm_datases/quirky_sciq/pythia-12b_results
Monotonicity: 0.000


Map:   0%|          | 0/11679 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format: 0ba [00:00, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format: 0ba [00:00, ?ba/s]

KeyboardInterrupt: 

In [122]:
import torch
dataset = "quirky_sciq_pythia-410m"
log_odds = torch.load(f"experiments/Mistral-7b-v0.1_{dataset}_bob/BH/test/BE_mean-diff_log_odds.pt", map_location="cpu")
bob_labels = torch.load(f"experiments/Mistral-7b-v0.1_{dataset}_bob/BH/test/bob_labels.pt", map_location="cpu")
alice_labels = torch.load(f"experiments/Mistral-7b-v0.1_{dataset}_bob/BH/test/alice_labels.pt", map_location="cpu")
lm_log_odds = torch.load(f"experiments/Mistral-7b-v0.1_{dataset}_bob/BH/test/lm_log_odds.pt", map_location="cpu")

In [123]:
from datasets import load_dataset

ds = load_dataset(f"atmallen/{dataset}_bob_easy", split="test").with_format("numpy")

In [124]:
(ds["bob_label"] == ds["alice_label"]).mean()

1.0

In [125]:
import numpy as np
from collections import Counter
Counter(ds["difficulty"]).most_common(5)

[(2.4219192e-12, 2),
 (4.795828e-21, 2),
 (9.7349895e-15, 2),
 (5.572718e-12, 2),
 (5.5111666e-13, 2)]

In [126]:
((lm_log_odds > 0) == bob_labels).float().mean()

tensor(0.4250)

In [127]:
(alice_labels == bob_labels).float().mean()

tensor(0.3850)

In [131]:
from sklearn.metrics import roc_auc_score
((lm_log_odds > 0) == bob_labels).float().mean(), roc_auc_score(bob_labels, lm_log_odds.float())

(tensor(0.4250), 0.4580704126602564)